**IMPORTATION**

In [23]:
from dataclasses import dataclass
import pickle
import gzip
import random
import numpy as np
from PIL import Image, ImageOps
from time import time

**Sigmoid Function**

In [24]:
def sigmoid(z):
  return 1.0 / (1.0 + np.exp(-z))

#def sigmoid_prime(z):
 # return sigmoid(z) * (1 - sigmoid(z))

Derivation

In [25]:
def sigmoid_prime(z):
  return sigmoid(z) * (1 - sigmoid(z))

To Identify Cost Function


In [26]:
def cost_derivative(output_activations, y):
  return (output_activations - y)

In [27]:
@dataclass
class Network:
  num_layers: int
  biases: list
  weights: list

def init_network(layers):
    return Network(
        len(layers),
        [np.random.randn(y, 1) for y in layers[1:]],
        [np.random.randn(y, x) for x, y in zip(layers[:-1], layers[1:])]
    )

**Forward propagation function**

In [28]:
def feedforward(nn, a):
  for b, w in zip(nn.biases, nn.weights):
    a = sigmoid(np.dot(w,a) + b)
  return a

In [29]:
def evaluate(nn, test_data):
  test_results = [(np.argmax(feedforward(nn, x)), y) for (x, y) in test_data]
  return sum(int(x == y) for (x, y) in test_results)


In [30]:
def learn(nn, training_data, epochs, mini_batch_size, learning_rate, test_data = None):
    n = len(training_data)

    for j in range(epochs):
        random.shuffle(training_data) # that's where "stochastic" comes from

        mini_batches = [
            training_data[k: k + mini_batch_size] for k in range(0, n, mini_batch_size)
        ]

        for mini_batch in mini_batches:
            stochastic_gradient_descent(nn, mini_batch, learning_rate) # that's where learning really happes

        if test_data:
            print('Epoch {0}: accuracy {1}%'.format(f'{j + 1:2}', 100.0 * evaluate(nn, test_data) / len(test_data)))
        else:
            print('Epoch {0} complete.'.format(f'{j + 1:2}'))

In [45]:
def stochastic_gradient_descent(nn, mini_batch, learning_rate):
    # Prepare mini_batch into X and Y matrices
    # X: matrix where each column is an input vector
    # Y: matrix where each column is a one-hot encoded label vector
    X = np.hstack([x for x, y in mini_batch])
    Y = np.hstack([y for x, y in mini_batch])

    nabla_b, nabla_w = backprop(nn, X, Y)

    # Update weights and biases using the averaged gradients
    nn.weights = [w - (learning_rate / len(mini_batch)) * n_w
                    for w, n_w in zip(nn.weights, nabla_w)]
    nn.biases = [b - (learning_rate / len(mini_batch)) * n_b
                   for b, n_b in zip(nn.biases, nabla_b)]

In [44]:
def backprop(nn, X, Y): # X, Y are matrices representing a mini-batch
    nabla_b = [np.zeros(b.shape) for b in nn.biases]
    nabla_w = [np.zeros(w.shape) for w in nn.weights]

    # Feedforward
    activation = X
    activations = [X] # list to store all activations, layer by layer
    zs = []           # list to store all z vectors, layer by layer

    for b, w in zip(nn.biases, nn.weights):
        z = np.dot(w, activation) + b  # calculate z for the current layer (bias b broadcasts)
        zs.append(z)
        activation = sigmoid(z)
        activations.append(activation)

    # Backward pass

    # 1. Output layer error
    delta = cost_derivative(activations[-1], Y) * sigmoid_prime(zs[-1])
    nabla_b[-1] = np.sum(delta, axis=1, keepdims=True) # Sum deltas across mini-batch examples
    nabla_w[-1] = np.dot(delta, activations[-2].transpose())

    # 2. Backpropagate through hidden layers
    for i in range(2, nn.num_layers):
        z = zs[-i]
        sp = sigmoid_prime(z)
        delta = np.dot(nn.weights[-i + 1].transpose(), delta) * sp

        nabla_b[-i] = np.sum(delta, axis=1, keepdims=True)
        nabla_w[-i] = np.dot(delta, activations[-i - 1].transpose())

    return (nabla_b, nabla_w)

MNIST data

In [33]:
def load_data():
    f = open('mnist.pkl', 'rb') # Changed from gzip.open to open
    training_data, validation_data, test_data = pickle.load(f, encoding="bytes")
    f.close()

    return (training_data, validation_data, test_data)

In [34]:
def load_data_wrapper():
    tr_d, va_d, te_d = load_data()

    training_inputs = [np.reshape(x, (784, 1)) for x in tr_d[0]]
    training_results = [one_hot_encode(y) for y in tr_d[1]]
    training_data = zip(training_inputs, training_results)
    validation_inputs = [np.reshape(x, (784, 1)) for x in va_d[0]]
    validation_data = zip(validation_inputs, va_d[1])
    test_inputs = [np.reshape(x, (784, 1)) for x in te_d[0]]
    test_data = zip(test_inputs, te_d[1])

    return (list(training_data), list(validation_data), list(test_data))

In [35]:
def one_hot_encode(j):
  e = np.zeros((10, 1))
  e[j] = 1.0
  return e

Utility Function

In [36]:
def print_shape(name, data):
  print('Shape of {0}: {1}'.format(name, data.shape))

Main Program

In [37]:
training_data, validation_data, test_data = load_data_wrapper() # load data

nn = init_network([784, 30, 10])

for l in range(0, nn.num_layers - 1):
    print('\nNetwork layer {0}'.format(l + 2)) # disregard the input layer
    print_shape('weights', nn.weights[l])
    print_shape('biases', nn.biases[l])

# hyper parameters
epochs = 15
mini_batch_size = 10
learning_rate = 3.0

print('\nLearning process started...\n')

time_start = time()

learn(nn, training_data, epochs, mini_batch_size, learning_rate, test_data)

time_end = time()

time_elapsed = time_end - time_start

print('\nLearning process complete in {0} seconds ({1} seconds per epoch)!\n'.format(f'{time_elapsed:.0f}', f'{time_elapsed / epochs:.1f}'))

print('Validation (with yet unseen data): accuracy {0}%'.format(100.0 * evaluate(nn, validation_data) / len(validation_data)))


Network layer 2
Shape of weights: (30, 784)
Shape of biases: (30, 1)

Network layer 3
Shape of weights: (10, 30)
Shape of biases: (10, 1)

Learning process started...

Epoch  1: accuracy 89.88%
Epoch  2: accuracy 91.93%
Epoch  3: accuracy 92.54%
Epoch  4: accuracy 92.94%
Epoch  5: accuracy 93.3%
Epoch  6: accuracy 93.46%
Epoch  7: accuracy 93.86%
Epoch  8: accuracy 93.67%
Epoch  9: accuracy 94.08%
Epoch 10: accuracy 94.03%
Epoch 11: accuracy 93.76%
Epoch 12: accuracy 94.13%
Epoch 13: accuracy 94.53%
Epoch 14: accuracy 94.42%
Epoch 15: accuracy 94.22%

Learning process complete in 104 seconds (6.9 seconds per epoch)!

Validation (with yet unseen data): accuracy 94.71%


Load Image Function

In [38]:
from PIL import Image, ImageOps

In [39]:
def load_image(file_name):
    digit = Image.open(file_name)

    # invert, so that background is black (zeros)
    digit = ImageOps.invert(digit)

    pixels = digit.load()

    return np.array(digit).reshape((784, 1)) / 255

In [40]:
def recognize_image(path, file):
    x = load_image(path.format(file))

    y = feedforward(nn, x)

    bitmap = x.reshape((28, 28))

    file_num = int(file)
    result = y.argmax()

    if file_num == result:
        ev = 'correctly'
    else:
        ev = 'incorrectly'

    print(file, 'was', ev, 'recognized as', result)

Real-life test!

In [41]:
print('Non-MNIST digits:\n')

for file in range(0,10):
    recognize_image('/0.png', 0)

Non-MNIST digits:

0 was correctly recognized as 0
0 was correctly recognized as 0
0 was correctly recognized as 0
0 was correctly recognized as 0
0 was correctly recognized as 0
0 was correctly recognized as 0
0 was correctly recognized as 0
0 was correctly recognized as 0
0 was correctly recognized as 0
0 was correctly recognized as 0


In [47]:
print('Non-MNIST digits:\n')

for file in range(0,10):
    recognize_image('/1.png', 1)

Non-MNIST digits:

1 was correctly recognized as 1
1 was correctly recognized as 1
1 was correctly recognized as 1
1 was correctly recognized as 1
1 was correctly recognized as 1
1 was correctly recognized as 1
1 was correctly recognized as 1
1 was correctly recognized as 1
1 was correctly recognized as 1
1 was correctly recognized as 1


Note: The previous cell `bTlnzImTbSAH` had a loop that was attempting to recognize the same image multiple times, and an incorrect path. I've removed that cell's code to avoid confusion and replaced it with a clearer, single-recognition attempt in the new cell above.

In [42]:
print('Recognizing image /content/0.png:')
recognize_image('/content/{}.png', 0)

Recognizing image /content/0.png:
0 was correctly recognized as 0


Before running the next cell, please ensure the import cell (`vDczJAyhtctE`) containing `import numpy as np` has been executed to resolve the `NameError`.

Let's test the network with your uploaded image `/content/0.png`.

**Assignment**

Now, let's test the new `Network` class with the `mnist.pkl` dataset.

In [54]:
import numpy as np

# Create 50,000 random training samples
training_data = []

for _ in range(50000):
    x = np.random.rand(784, 1)      # Input (MNIST-sized)
    y = np.zeros((10, 1))           # One-hot output

    label = np.random.randint(0, 10)
    y[label] = 1.0

    training_data.append((x, y))

# Create network
net = Network([784, 30, 10])

# Train
net.SGD(
    training_data=training_data,
    epochs=5,
    mini_batch_size=100,
    eta=3.0
)

print("Training completed on 50,000 samples.")

Epoch 0 complete
Epoch 1 complete
Epoch 2 complete
Epoch 3 complete
Epoch 4 complete
Training completed on 50,000 samples.


In [55]:
import time

start = time.time()

net.SGD(
    training_data=training_data,
    epochs=5,
    mini_batch_size=100,
    eta=3.0
)

end = time.time()

print("Execution Time:", end - start, "seconds")

Epoch 0 complete
Epoch 1 complete
Epoch 2 complete
Epoch 3 complete
Epoch 4 complete
Execution Time: 3.931232452392578 seconds


In [56]:
import numpy as np
import time

# Sigmoid
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_prime(z):
    s = sigmoid(z)
    return s * (1 - s)

# Network dimensions
input_size = 784
hidden_size = 128
output_size = 10

# 50,000 random samples
m = 50000

# Random data
X = np.random.randn(input_size, m)
Y = np.random.randn(output_size, m)

# Random weights
W1 = np.random.randn(hidden_size, input_size) * 0.01
b1 = np.zeros((hidden_size, 1))

W2 = np.random.randn(output_size, hidden_size) * 0.01
b2 = np.zeros((output_size, 1))

start = time.time()

# ======================
# FORWARD PROPAGATION
# ======================

Z1 = W1 @ X + b1
A1 = sigmoid(Z1)

Z2 = W2 @ A1 + b2
A2 = sigmoid(Z2)

# ======================
# BACKPROPAGATION
# ======================

delta2 = (A2 - Y) * sigmoid_prime(Z2)

dW2 = (delta2 @ A1.T) / m
db2 = np.sum(delta2, axis=1, keepdims=True) / m

delta1 = (W2.T @ delta2) * sigmoid_prime(Z1)

dW1 = (delta1 @ X.T) / m
db1 = np.sum(delta1, axis=1, keepdims=True) / m

end = time.time()

print("Samples:", m)
print("Forward + Backward Time:", end - start, "seconds")

print("dW1 shape =", dW1.shape)
print("db1 shape =", db1.shape)
print("dW2 shape =", dW2.shape)
print("db2 shape =", db2.shape)

Samples: 50000
Forward + Backward Time: 1.1010441780090332 seconds
dW1 shape = (128, 784)
db1 shape = (128, 1)
dW2 shape = (10, 128)
db2 shape = (10, 1)
